# Option C Test: Japanese Works Without CJK Stripping

Tests whether the multilingual BERT model (`bert-base-multilingual-cased`) can classify
Japanese works more accurately when we **keep** the original CJK/Hiragana/Katakana text
instead of stripping it to leftover Latin fragments.

**Background**: The current pipeline strips all non-Latin characters before classification.
For Japanese articles, this leaves garbage Latin fragments that the model misclassifies
(commonly as "Military Technology and Strategies", topic t14423).

Since `bert-base-multilingual-cased` was trained on 104 languages including Japanese,
it may actually produce better predictions on the original text.

**Sample size**: 50,000 IRDB works currently classified as t14423
**Run on**: GPU cluster (topics-gpu-cluster)

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch
import pandas as pd

MODEL_PATH = "/Volumes/openalex/works/models/topic-classification-title-abstract"
BATCH_SIZE = 25

classifier = pipeline(
    task="text-classification",
    model=MODEL_PATH,
    device=0 if torch.cuda.is_available() else -1,
    top_k=3,
    batch_size=BATCH_SIZE,
    truncation=True,
    max_length=512
)

print(f"Model loaded on: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# Pull 50k IRDB works currently classified as t14423 (Military Technology)
# Source s7407056385 = IRDB
sample_df = spark.sql("""
    SELECT w.id as work_id, w.title, w.abstract,
           w.primary_location.source.display_name as journal_name,
           w.topics[0].id as current_topic_id,
           w.topics[0].display_name as current_topic_name,
           w.topics[0].score as current_topic_score
    FROM openalex.works.openalex_works_base w
    WHERE w.primary_topic.id = 'https://openalex.org/T14423'
      AND EXISTS(
          SELECT 1 FROM explode(w.locations) loc
          WHERE loc.col.source.id = 'https://openalex.org/S7407056385'
      )
    LIMIT 50000
""").toPandas()

print(f"Sample size: {len(sample_df)}")
sample_df[['work_id', 'title', 'current_topic_name', 'current_topic_score']].head(10)

In [ ]:
from topic_predictor import clean_title, clean_abstract, remove_non_latin_characters, is_heavily_stripped
import re

def format_stripped(title, abstract):
    """Current approach: strip all non-Latin characters"""
    t = clean_title(title) or ""
    a = clean_abstract(abstract) or ""
    return f"[CLS]<TITLE> {t.strip()} <ABSTRACT> {a.strip()} [SEP]"

def format_unstripped(title, abstract):
    """Option C: keep original Japanese text, only clean HTML"""
    t = title.strip() if isinstance(title, str) and title else ""
    # Basic HTML tag removal (same tags as clean_title but without character stripping)
    if t and '<' in t:
        t = re.sub(r'<[^>]+>', '', t)
    a = abstract.strip()[:2500] if isinstance(abstract, str) and abstract and len(abstract) > 10 else ""
    return f"[CLS]<TITLE> {t} <ABSTRACT> {a} [SEP]"

# Build both versions of input text
stripped_texts = []
unstripped_texts = []

# Also track which works would be caught by Option B
option_b_skipped = []

for _, row in sample_df.iterrows():
    stripped_texts.append(format_stripped(row['title'], row['abstract']))
    unstripped_texts.append(format_unstripped(row['title'], row['abstract']))
    
    # Check Option B gating
    cleaned_title = clean_title(row['title']) or ""
    cleaned_abstract = clean_abstract(row['abstract']) if row['abstract'] else None
    title_stripped = is_heavily_stripped(row['title'], cleaned_title)
    abstract_stripped = is_heavily_stripped(row['abstract'], cleaned_abstract)
    option_b_skipped.append(title_stripped and (abstract_stripped or not row['abstract']))

sample_df['option_b_skipped'] = option_b_skipped

print(f"Total works: {len(sample_df)}")
print(f"Would be skipped by Option B: {sum(option_b_skipped)} ({sum(option_b_skipped)/len(sample_df)*100:.1f}%)")
print(f"\n=== Example comparison ===")
print(f"Original title:   {sample_df.iloc[0]['title'][:100]}")
print(f"Stripped input:   {stripped_texts[0][:100]}")
print(f"Unstripped input: {unstripped_texts[0][:100]}")

import time

# Run predictions with both approaches — batch to handle 50k
print("Running stripped predictions (current approach)...")
start = time.time()
stripped_preds = classifier(stripped_texts)
print(f"  Done in {time.time() - start:.1f}s")

print("Running unstripped predictions (Option C)...")
start = time.time()
unstripped_preds = classifier(unstripped_texts)
print(f"  Done in {time.time() - start:.1f}s")

In [ ]:
# Build comparison dataframe
results = []
for i, (_, row) in enumerate(sample_df.iterrows()):
    stripped_top = stripped_preds[i][0] if stripped_preds[i] else {}
    unstripped_top = unstripped_preds[i][0] if unstripped_preds[i] else {}
    
    stripped_id = 10000 + int(stripped_top['label'].split(':')[0]) if stripped_top.get('label') else None
    unstripped_id = 10000 + int(unstripped_top['label'].split(':')[0]) if unstripped_top.get('label') else None
    
    results.append({
        'work_id': row['work_id'],
        'title': row['title'][:80] if row['title'] else None,
        'option_b_skipped': row['option_b_skipped'],
        'stripped_topic_id': stripped_id,
        'stripped_label': stripped_top.get('label', '').split(': ', 1)[-1] if stripped_top.get('label') else None,
        'stripped_score': round(stripped_top.get('score', 0), 4),
        'unstripped_topic_id': unstripped_id,
        'unstripped_label': unstripped_top.get('label', '').split(': ', 1)[-1] if unstripped_top.get('label') else None,
        'unstripped_score': round(unstripped_top.get('score', 0), 4),
        'changed': stripped_id != unstripped_id
    })

results_df = pd.DataFrame(results)

print(f"Total works: {len(results_df)}")
print(f"Topic changed (stripped vs unstripped): {results_df['changed'].sum()} ({results_df['changed'].mean()*100:.1f}%)")
print(f"\nStill Military Tech after unstripping: {(results_df['unstripped_topic_id'] == 14423).sum()}")
print(f"Still Military Tech with stripping (sanity check): {(results_df['stripped_topic_id'] == 14423).sum()}")
print(f"\nWould be skipped by Option B: {results_df['option_b_skipped'].sum()} ({results_df['option_b_skipped'].mean()*100:.1f}%)")

In [ ]:
# Run predictions with both approaches
print("Running stripped predictions (current approach)...")
stripped_preds = classifier(stripped_texts)

print("Running unstripped predictions (Option C)...")
unstripped_preds = classifier(unstripped_texts)

print("Done.")

In [ ]:
# Show side-by-side comparison for a sample of works where the topic changed
changed_df = results_df[results_df['changed']].copy()
print(f"\n=== {len(changed_df)} works where topic changed ===")
pd.set_option('display.max_colwidth', 80)
changed_df[['title', 'stripped_label', 'stripped_score', 'unstripped_label', 'unstripped_score']].head(30)

In [ ]:
# Show distribution of unstripped predictions
print("=== Top predicted topics when Japanese text is KEPT ===")
print(results_df['unstripped_label'].value_counts().head(20))
print("\n=== Top predicted topics when Japanese text is STRIPPED (current) ===")
print(results_df['stripped_label'].value_counts().head(20))

In [ ]:
# Save full results to Delta table for later analysis
results_spark = spark.createDataFrame(results_df)
results_spark.write.mode("overwrite").saveAsTable("openalex.works.topic_japanese_test_results")
print(f"Saved {len(results_df)} results to openalex.works.topic_japanese_test_results")

In [ ]:
# Show side-by-side comparison for works where the topic changed
changed_df = results_df[results_df['changed']].copy()
print(f"\n=== {len(changed_df)} works where topic changed ===")
pd.set_option('display.max_colwidth', 80)
changed_df[['title', 'stripped_label', 'stripped_score', 'unstripped_label', 'unstripped_score']].head(30)

In [ ]:
# Score confidence comparison
print("=== Average confidence scores ===")
print(f"Stripped (current):   {results_df['stripped_score'].mean():.4f}")
print(f"Unstripped (Option C): {results_df['unstripped_score'].mean():.4f}")
print(f"\nStripped score distribution:")
print(results_df['stripped_score'].describe())
print(f"\nUnstripped score distribution:")
print(results_df['unstripped_score'].describe())